## 데이터 전처리 파이프라인 주요 수정 및 고도화 내역
### 텍스트 정제 및 청킹 프로세스 통합함
- 기존에는 특수문자 등을 제거하는 정제 단계(Preprocessing_Cleaning.ipynb)와 텍스트를 분할하는 단계(text_chunking.ipynb)가 분리되어 있었음.

- clean_and_chunk_with_meta 함수를 생성하여, 텍스트 정규표현식 정제 후 곧바로 길이 기반 청킹이 이루어지도록 단일 파이프라인으로 통합함.

### 청크(Chunk) 단위 메타데이터 주입 기능 추가함
- 쪼개진 텍스트 조각만으로는 AI가 맥락을 파악하기 어렵다는 점을 보완함.

- 각 조각 최상단에 [발주기관 | 사업명 | 금액] 형태의 메타데이터(이름표)를 강제 주입하는 로직을 추가함.

### 메타데이터 세부 정보 및 안정성 고도화함
- 결측치(NaN) 방어 로직 적용: 데이터가 비어있을 경우 'nan' 텍스트나 에러가 발생하는 것을 막기 위해, 빈 값을 '미상'으로 안전하게 치환하는 safe_str 헬퍼 함수를 적용함.

- 핵심 검색 단서 추가: 복구했던 데이터인 '공고 번호'와 '입찰 참여 시작일'을 메타데이터 항목에 추가하여 검색 정밀도를 높임.

- 조각 순서(Index) 표기: 분할된 문서의 전후 맥락 유지를 위해 조각순서: 1/5 형태로 현재 청크의 위치 정보를 메타데이터에 포함함.

### 구조 기반 청킹(심화) 적용 보류함
- 공통 섹션 구조(목차 패턴 등)를 정규표현식으로 인식하여 자르는 '구조 기반 청킹'은 현 단계의 오버엔지니어링으로 판단하여 제외함.

- 기존에 작성한 글자 수 기반의 RecursiveCharacterTextSplitter 방식을 유지하여 빠르고 안정적인 분할을 우선시함.

In [1]:
import re
import pandas as pd
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm

print("🚀 [Step 2] 정보 보존형 정제 ➡️ 청킹 ➡️ 메타데이터 주입 파이프라인 시작...\n")

# ./data_list.csv
# 1. 원본 데이터 불러오기
file_path = './bid_master_cleaned.csv'  # 실제 환경에 맞는 경로로 수정하세요
df = pd.read_csv(file_path)
print(f"✅ 원본 데이터 로드 완료: 총 {len(df)}건\n")

# 2. 통합 처리 함수 정의
def clean_and_chunk_with_meta(row):
    raw_text = row.get('텍스트', '')
    
    # 데이터가 없거나 빈 경우 빈 리스트 반환
    if pd.isna(raw_text) or len(str(raw_text).strip()) == 0:
        return []
    
    # ------------------------------------------------------------
    # 🧹 [Step 1] 정보 보존형 텍스트 정제
    # ------------------------------------------------------------
    # 정보 유실을 막기 위해 따옴표, 불렛 기호(■, ※, ○), 수학 기호 등을 대거 허용
    allowed_chars = r'[^가-힣a-zA-Z0-9\s\.\(\)\[\]\/\,\%\:\-\·\?\!\@\+\=\<\>\~\&\*\"\'\■\※\•\○\●\·\_\<\>\=\㎡\㎥\원]'
    cleaned_text = re.sub(allowed_chars, ' ', str(raw_text))
    
    # 불필요한 연속 공백 제거
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    cleaned_len = len(cleaned_text)
    
    # ------------------------------------------------------------
    # ✂️ [Step 2] 문장 맥락 보존형 청킹
    # ------------------------------------------------------------
    # 문서 길이에 따른 가변 사이즈 전략
    if cleaned_len < 500:
        chunk_size, chunk_overlap = 500, 0
    elif cleaned_len < 3000:
        chunk_size, chunk_overlap = 600, 100
    else:
        chunk_size, chunk_overlap = 1000, 200

    # separators 순서를 조정하여 문장이 최대한 원형대로 보존되게 함
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", "? ", "! ", "; ", " ", ""]
    )
    chunks = splitter.split_text(cleaned_text)
    
    # ------------------------------------------------------------
    # 🏷️ [Step 3] 메타데이터 주입 및 안전한 문자열 처리
    # ------------------------------------------------------------
    def safe_str(val):
        return "미상" if pd.isna(val) or str(val).strip() == "" else str(val).strip()
    
    # 금액 포맷팅 (숫자형이 아닐 경우 대비 예외 처리)
    try:
        amt_val = row.get('사업 금액')
        amt_str = f"{float(amt_val):,.0f}원" if not pd.isna(amt_val) else "미상"
    except (ValueError, TypeError):
        amt_str = safe_str(row.get('사업 금액'))
        
    notice_id = safe_str(row.get('공고 번호'))
    start_date = safe_str(row.get('입찰 참여 시작일'))
    agency = safe_str(row.get('발주 기관'))
    project_name = safe_str(row.get('사업명'))
    
    total_chunks = len(chunks)
    chunks_with_meta = []
    
    for i, chunk_text in enumerate(chunks):
        # 검색 시 문맥 식별을 돕는 헤더 구성
        meta_header = (
            f"[공고번호: {notice_id} | 발주기관: {agency} | "
            f"사업명: {project_name} | 금액: {amt_str} | "
            f"시작일: {start_date} | 조각순서: {i+1}/{total_chunks}]"
        )
        # 헤더와 본문을 분리하여 저장
        chunk_with_meta = f"{meta_header}\n\n{chunk_text}"
        chunks_with_meta.append(chunk_with_meta)
        
    return chunks_with_meta

# 3. 파이프라인 실행
print("✂️ 정제, 청킹, 메타데이터 주입을 진행합니다...")
tqdm.pandas()
df['청크_리스트'] = df.progress_apply(clean_and_chunk_with_meta, axis=1)

# 4. 행 분리 (Explode)
chunked_df = df.explode('청크_리스트').reset_index(drop=True)
chunked_df = chunked_df.rename(columns={'청크_리스트': '청크_텍스트'})

# 5. 불필요 컬럼 제거 및 길이 계산
chunked_df = chunked_df.drop(columns=['텍스트', '텍스트길이'], errors='ignore')
chunked_df['청크_길이'] = chunked_df['청크_텍스트'].apply(lambda x: len(x) if isinstance(x, str) else 0)

# 6. 최종 데이터 저장
save_path = './bid_master_chunked.csv'
chunked_df.to_csv(save_path, index=False, encoding='utf-8-sig')

print("-" * 60)
print(f"✨ 작업 완료! 원본 {len(df)}건 -> 총 {len(chunked_df)}개의 청크 생성.")
print(f"💾 결과 저장 완료: '{save_path}'")

# 결과 샘플 확인
display(chunked_df[['사업명', '청크_텍스트', '청크_길이']].head())

🚀 [Step 2] 정보 보존형 정제 ➡️ 청킹 ➡️ 메타데이터 주입 파이프라인 시작...

✅ 원본 데이터 로드 완료: 총 100건

✂️ 정제, 청킹, 메타데이터 주입을 진행합니다...


100%|██████████| 100/100 [00:00<00:00, 749.00it/s]


------------------------------------------------------------
✨ 작업 완료! 원본 100건 -> 총 685개의 청크 생성.
💾 결과 저장 완료: './bid_master_chunked.csv'


,사업명,청크_텍스트,청크_길이
0,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화,[공고번호: 20241001798 | 발주기관: 한영대학 | 사업명: 한영대학교 특...,471
1,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화,[공고번호: 20241001798 | 발주기관: 한영대학 | 사업명: 한영대학교 특...,716
2,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화,[공고번호: 20241001798 | 발주기관: 한영대학 | 사업명: 한영대학교 특...,309
3,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화,[공고번호: 20241001798 | 발주기관: 한영대학 | 사업명: 한영대학교 특...,302
4,2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선,[공고번호: 20241002912 | 발주기관: 한국연구재단 | 사업명: 2024년...,544
